In [ ]:
import sys
sys.path.append('../../src')
import src.EmbeddingBase
import numpy as np
from sklearn.neighbors import NearestNeighbors
from typing import Dict, List, Tuple, Optional

def cohen_procaccia_entropy(
    emb: EmbeddingBase,
    eps: float,
    m_list: List[int],
    tau: float = 1.0,
    theiler: int = 0,
    log_base: float = np.e,   # use 2 for bits/s, np.e for nats/s
    max_neighbors: Optional[int] = None,
) -> Dict[str, Dict[int, float]]:
    """
    Estimate H_m(eps) and h_CP(eps, tau) from trajectories via the Cohen–Procaccia procedure.

    Parameters
    ----------
    emb : EmbeddingBase
        Your embedding object with emb.Y of shape (N, T, d) containing N trajectories.
    eps : float
        Spatial resolution (radius in Chebyshev norm) used in the CP estimator.
    m_list : list[int]
        Block lengths m (delay window K) to evaluate.
    tau : float
        Sampling interval of the time series.
    theiler : int
        Theiler window (in time steps) to exclude temporally adjacent pairs within each trajectory.
    log_base : float
        Logarithm base for entropies (np.e → nats; 2 → bits).
    max_neighbors : Optional[int]
        Optional cap to speed up neighbor queries (passed to sklearn).

    Returns
    -------
    dict with keys:
        "H_m":  dict mapping m -> H_m(eps)
        "h_cp": dict mapping m -> h_CP(eps, tau) ≈ [H_{m+1}-H_m]/tau  (only for m with m+1 in m_list)
    """
    if emb.Y is None:
        raise RuntimeError("EmbeddingBase must contain trajectories in emb.Y")

    N, T, d = emb.Y.shape
    Hm: Dict[int, float] = {}

    # helper to map flattened index -> (traj_index, window_start)
    def idx_to_nt(idx: int, L: int) -> Tuple[int, int]:
        n = idx // L
        t0 = idx - n * L
        return n, t0

    for m in sorted(m_list):
        # 1) Build m-delay embedding over all trajectories using your API
        emb.make_embedding(K=m)  # sets emb.L = T-m+1, emb.flatten_embedding_matrix shape (N*L, m*d)
        L = emb.L
        X = emb.flatten_embedding_matrix  # shape (N*L, m*d)
        M = X.shape[0]

        # 2) Radius-neighbors in Chebyshev (∞-norm) on the m*d vectors
        # CP uses max over time (and often over coordinates) → Chebyshev on the flattened window matches that.
        nn = NearestNeighbors(
            radius=eps, metric="chebyshev", algorithm="auto"
        )
        nn.fit(X)
        # For speed, you can set max_neighbors if desired (sklearn 1.4+)
        neigh_ind = nn.radius_neighbors(return_distance=False, sort_results=False, X=X)

        # 3) Count valid neighbors per point excluding:
        #    - self
        #    - points from the SAME trajectory with |Δt| < theiler
        counts = np.empty(M, dtype=int)
        for i in range(M):
            n_i, t_i = idx_to_nt(i, L)
            neigh = neigh_ind[i]

            # drop self if present
            neigh = neigh[neigh != i]

            if theiler > 0:
                # filter out neighbors from same trajectory within Theiler window
                keep = []
                for j in neigh:
                    n_j, t_j = idx_to_nt(j, L)
                    if n_j != n_i or abs(t_j - t_i) >= theiler:
                        keep.append(j)
                neigh = np.array(keep, dtype=int)

            counts[i] = neigh.size

        # 4) CP block entropy estimate:
        #    H_m(eps) = - (1/M) * sum_i log( counts_i / (M_eff - 1) )
        #    with M_eff = M - 1 - (#excluded within Theiler for point i)
        # Here we approximate M_eff by (M - 1) since exclusions are negligible in long signals;
        # alternatively, you can compute a per-i normalizer by explicitly counting excluded indices.
        normalizer = (M - 1)
        # avoid zeros for log; set minimal count to 1 to keep estimator finite
        safe_counts = np.maximum(counts, 1)
        H_m = -np.mean(np.log(safe_counts / normalizer)) / np.log(log_base)
        Hm[m] = H_m

    # 5) Entropy-rate estimate: h_CP ≈ [H_{m+1} - H_m] / tau
    hcp: Dict[int, float] = {}
    m_sorted = sorted(m_list)
    for i in range(len(m_sorted) - 1):
        m = m_sorted[i]
        mp1 = m_sorted[i + 1]
        hcp[m] = (Hm[mp1] - Hm[m]) / tau

    return {"H_m": Hm, "h_cp": hcp}
